# Lab 02 — 加工結果の検証（読み取り専用）

保存済みの `green_tripdata_2026` を読み直し、全件の品質、加工ルール、固定レコード、集計を期待値と照合します。
**データは変更しません。** 不一致があっても修正せず、差分を報告します。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
raw = spark.read.parquet("Files/raw/green_tripdata_2026-04.parquet")
trips = spark.table("green_tripdata_2026")

check("source rows", 44238, raw.count())
check("source columns", 21, len(raw.columns))
check("saved rows", 37948, trips.count())
check("saved columns", 27, len(trips.columns))
check("ehail_fee removed", False, "ehail_fee" in trips.columns)
check("trip_id unique 1..37948", [37948, 1, 37948],
      list(trips.agg(F.countDistinct("trip_id"), F.min("trip_id"), F.max("trip_id")).first()))

quality = trips.agg(
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("null_passenger"),
    F.sum((~F.col("pickup_hour").between(0, 23) | F.col("pickup_hour").isNull()).cast("int")).alias("bad_hour"),
    F.sum((F.col("pickup_month") != "2026-04").cast("int")).alias("other_month"),
    F.round(F.sum("total_amount"), 2).alias("total_amount"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
).first()
check("passenger_count nulls", 0, quality.null_passenger)
check("pickup_hour out of range", 0, quality.bad_hour)
check("pickup_month other than 2026-04", 0, quality.other_month)
check("total_amount sum", 687602.60, float(quality.total_amount), 0.005)
check("average fare", 11.92, float(quality.avg_fare), 0.005)

## 派生列の計算ルールを全件で照合

In [ ]:
seconds = F.col("lpep_dropoff_datetime").cast("long") - F.col("lpep_pickup_datetime").cast("long")
recomputed = trips.select(
    (F.col("pickup_date") != F.to_date("lpep_pickup_datetime")).alias("date"),
    (F.col("pickup_hour") != F.hour("lpep_pickup_datetime")).alias("hour"),
    (~F.col("trip_duration_minutes").eqNullSafe(F.when(seconds >= 0, F.round(seconds / 60.0, 2)))).alias("duration"),
    (~F.col("fare_per_mile").eqNullSafe(F.when(F.col("trip_distance") > 0, F.round(F.col("fare_amount") / F.col("trip_distance"), 2)))).alias("fare_per_mile"),
    (F.col("passenger_group") != F.when(F.col("passenger_count") == 1, "single")
        .when(F.col("passenger_count") >= 2, "group").otherwise("unknown")).alias("group"),
)
mismatches = recomputed.agg(*[F.sum(F.col(c).cast("int")).alias(c) for c in recomputed.columns]).first().asDict()
for column, count in mismatches.items():
    check(f"derived column rule: {column}", 0, count or 0)

## 固定レコード（WHERE 条件で特定し、各条件の該当件数も確認）

In [ ]:
EXPECTED = {
    1: dict(VendorID=2, PULocationID=243, DOLocationID=108, passenger_count=5, trip_distance=6.64, fare_amount=23.07,
            total_amount=35.41, trip_duration_minutes=34.0, fare_per_mile=3.47, passenger_group="group",
            pickup="2026-04-01 00:00:02", dropoff="2026-04-01 00:34:02"),
    4: dict(VendorID=1, PULocationID=206, DOLocationID=89, passenger_count=1, trip_distance=5.94, fare_amount=15.83,
            total_amount=19.38, trip_duration_minutes=36.0, fare_per_mile=2.66, passenger_group="single",
            pickup="2026-04-01 00:01:51", dropoff="2026-04-01 00:37:51"),
}
fixed = (
    trips.filter((F.col("pickup_date") == F.lit("2026-04-01").cast("date")) & (F.col("pickup_hour") == 0) & (
        ((F.col("VendorID") == 1) & (F.col("PULocationID") == 206) & (F.col("DOLocationID") == 89))
        | ((F.col("VendorID") == 2) & (F.col("PULocationID") == 243) & (F.col("DOLocationID") == 108))))
    .withColumn("pickup", F.date_format("lpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("dropoff", F.date_format("lpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss"))
    .orderBy("trip_id")
)
rows = {row.trip_id: row.asDict() for row in fixed.collect()}
check("fixed records found (trip_id)", sorted(EXPECTED), sorted(rows))
for trip_id, expected in EXPECTED.items():
    actual = rows.get(trip_id, {})
    check(f"trip {trip_id}", expected, {key: actual.get(key) for key in expected})
fixed.select("trip_id", "VendorID", "PULocationID", "DOLocationID", "pickup", "dropoff", "passenger_count",
             "trip_distance", "fare_amount", "trip_duration_minutes", "fare_per_mile", "passenger_group").show(truncate=False)

## 集計（全件から計算）

In [ ]:
groups = {r.passenger_group: r["count"] for r in trips.groupBy("passenger_group").count().collect()}
check("passenger_group counts", {"group": 31691, "single": 6257}, groups)
check("passenger_group total", 37948, sum(groups.values()))
hours = trips.groupBy("pickup_hour").agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
check("hourly total", 37948, hours.agg(F.sum("trips")).first()[0])
hour0 = hours.filter("pickup_hour = 0").first()
check("hour 0", {"trips": 1615, "avg_fare": 11.94}, {"trips": hour0.trips, "avg_fare": float(hour0.avg_fare)})

failed = [c for c in checks if not c["passed"]]
save_result("nb_02_validate_trips", {"passed": not failed, "failed": len(failed), "checks": checks})